# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mustafaelsayedk71-sys/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [6]:

!git clone https://github.com/mustafaelsayedk71-sys/flyrank-ml-internship


%cd https://github.com/mustafaelsayedk71-sys/flyrank-ml-internship

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 127, done.
remote: Counting objects: 100% (127/127), done.
remote: Compressing objects: 100% (82/82), done.
remote: Total 127 (delta 39), reused 101 (delta 29), pack-reused 0 (from 0)
Receiving objects: 100% (127/127), 1.82 MiB | 4.40 MiB/s, done.
Resolving deltas: 100% (39/39), done.
[Errno 2] No such file or directory: 'https://github.com/mustafaelsayedk71-sys/flyrank-ml-internship'
/content


In [5]:
# Unit of Analysis: One row represents a unique content item / URL evaluated within a specific monthly time window.
# Time Window: Rolling 90-day window for feature aggregation evaluated on a mid-panel month (month = '2026-03').

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [4]:
# Features: impressions_90d, search_volume, ctr, clicks_90d (historical search performance).

# Label / Proxy: CTR_Opportunity_Score = impressions_90d * (1 - ctr) (measures rank potential).

# Context: url, month (identifiers for mapping and tracking).

# Excluded: Immediate post-refresh metrics, to strictly avoid target leakage during model training.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [7]:
import pandas as pd
import numpy as np

# 1. Load the dataset from raw path
data_path = 'flyrank-ml-internship/data/raw/content_refresh_anonymized.csv'
df = pd.read_csv(data_path)

# Query 1: Verify Grain & Shape
print("=== GRAIN VERIFICATION ===")
print("Dataset Shape:", df.shape)

# Query 2: Availability & Counts
print("\n=== COUNTS & MISSING VALUES ===")
print("Total rows:", len(df))
print("Available rows (impressions > 0):", (df['impressions_90d'] > 0).sum())
print("Missing CTR values:", df['ctr'].isnull().sum())

# Query 3: Target Definition & Leakage Trap Proof
df['CTR_Opportunity_Score'] = df['impressions_90d'] * (1 - df['ctr'])

# Create a intentional leaked feature to prove the trap
df['leaked_feature'] = df['CTR_Opportunity_Score'] * 0.99 + np.random.normal(0, 0.1, len(df))

print("\n=== TARGET & LEAKED FEATURE SAMPLE ===")
df[['search_volume', 'impressions_90d', 'ctr', 'CTR_Opportunity_Score', 'leaked_feature']].head()

=== GRAIN VERIFICATION ===
Dataset Shape: (30000, 44)

=== COUNTS & MISSING VALUES ===
Total rows: 30000
Available rows (impressions > 0): 30000
Missing CTR values: 0

=== TARGET & LEAKED FEATURE SAMPLE ===


,search_volume,impressions_90d,ctr,CTR_Opportunity_Score,leaked_feature
0,10.0,3803,0.76,912.72,903.564984
1,90.0,15320,0.05,14554.00,14408.313294
2,0.0,12581,0.09,11448.71,11334.153868
3,10.0,11751,0.49,5993.01,5933.197265
4,0.0,19140,0.13,16651.80,16485.234835


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [3]:
# Data Limitations: The dataset assumes steady baseline search demand. Sudden macro trends, seasonality, or Google core algorithm updates occurring within the prediction window are not captured in historical 90-day rolling metrics, creating potential noise for volatile search topics.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.